In [0]:
# ==============================================================================
# Pipeline Step: 01_bronze_to_silver.py
# Description: Ingests raw CSV and JSON datasets from the Bronze ADLS Gen2 container,
#              applies schema cleanups/deduplications, and saves them as Delta
#              tables into the Silver container.
# ==============================================================================

from pyspark.sql.functions import col, current_timestamp, to_timestamp
from pyspark.sql.types import DoubleType, IntegerType

# ------------------------------------------------------------------------------
# 1. Storage Account Credentials & Path Configurations
# ------------------------------------------------------------------------------
storage_account = "sttransitanalyticsdev"
storage_key = "kcLG0+ay1Ff8B2BXabvChxvASTlgkEiCwXMGdb4cjbEq3WanFq/u3uvrh9IldGJzwQbsgLaMwo+7+ASt/+3sJQ=="

# Define ADLS Gen2 ABFSS endpoints for Bronze and Silver layers
BRONZE_PATH = f"abfss://bronze@{storage_account}.dfs.core.windows.net"
SILVER_PATH = f"abfss://silver@{storage_account}.dfs.core.windows.net"

# Package explicit storage key credentials for PySpark access
storage_options = {
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net": storage_key
}

print("✅ Storage credentials and paths initialized successfully.")

# ------------------------------------------------------------------------------
# 2. Ingest Raw Datasets from Bronze Layer
# ------------------------------------------------------------------------------

# Load raw CSV files
df_buses_raw = (
    spark.read.options(**storage_options)
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{BRONZE_PATH}/buses.csv")
)

df_routes_raw = (
    spark.read.options(**storage_options)
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{BRONZE_PATH}/routes.csv")
)

df_passengers_raw = (
    spark.read.options(**storage_options)
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{BRONZE_PATH}/passengers.csv")
)

df_trips_raw = (
    spark.read.options(**storage_options)
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{BRONZE_PATH}/trips_01.csv")
)

df_payments_raw = (
    spark.read.options(**storage_options)
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{BRONZE_PATH}/payments.csv")
)

# Load raw multiline JSON file
df_events_raw = (
    spark.read.options(**storage_options)
    .option("multiline", "true")
    .json(f"{BRONZE_PATH}/vehicle_events.json")
)

print("✅ All 6 Bronze datasets successfully loaded into DataFrames.")

# ------------------------------------------------------------------------------
# 3. Transform & Persist Cleaned Data to Silver (Delta Format)
# ------------------------------------------------------------------------------

# --- 3.1 Clean Buses Data ---
df_buses_silver = (
    df_buses_raw
    .dropDuplicates(["bus_id"])
    .withColumn("capacity", col("capacity").cast(IntegerType()))
    .withColumn("ingested_at", current_timestamp())
)

df_buses_silver.write.options(**storage_options) \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/buses")

# --- 3.2 Clean Routes Data ---
df_routes_silver = (
    df_routes_raw
    .dropDuplicates(["route_id"])
    .withColumn("base_fare", col("base_fare").cast(DoubleType()))
    .withColumn("ingested_at", current_timestamp())
)

df_routes_silver.write.options(**storage_options) \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/routes")

# --- 3.3 Clean Passengers Data ---
df_passengers_silver = (
    df_passengers_raw
    .dropDuplicates(["passenger_id"])
    .withColumn("created_at", to_timestamp(col("created_at"), "yyyy-MM-dd"))
    .withColumn("ingested_at", current_timestamp())
)

df_passengers_silver.write.options(**storage_options) \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/passengers")

# --- 3.4 Clean Trips Data ---
df_trips_silver = (
    df_trips_raw
    .dropDuplicates(["trip_id"])
    .withColumn("start_time", to_timestamp(col("start_time"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("end_time", to_timestamp(col("end_time"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("passenger_count", col("passenger_count").cast(IntegerType()))
    .withColumn("ingested_at", current_timestamp())
)

df_trips_silver.write.options(**storage_options) \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/trips")

# --- 3.5 Clean Payments Data ---
df_payments_silver = (
    df_payments_raw
    .dropDuplicates(["payment_id"])
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("payment_timestamp", to_timestamp(col("payment_timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ingested_at", current_timestamp())
)

df_payments_silver.write.options(**storage_options) \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/payments")

# --- 3.6 Clean Vehicle Events Data ---
df_events_silver = (
    df_events_raw
    .dropDuplicates(["event_id"])
    .withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ingested_at", current_timestamp())
)

df_events_silver.write.options(**storage_options) \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/vehicle_events")

print("🚀 SUCCESS! All Bronze data cleaned and written as Delta tables into the Silver layer.")

✅ Storage credentials and paths initialized successfully.
✅ All 6 Bronze datasets successfully loaded into DataFrames.
🚀 SUCCESS! All Bronze data cleaned and written as Delta tables into the Silver layer.
